In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
import numpy as np
import pandas as pd
from pathlib import Path
from utils.benchmarks import BENCHMARKS_TO_PATHS


In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_0.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 0 ###

benchmark_name = "nyc-flight"
factor = 2
flights_df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "nyc_flights.csv"
)
flights_df = pd.concat([flights_df] * factor, ignore_index=True)

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_1.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 1 ###

flights_df.shape, flights_df.columns, flights_df.dtypes

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_2.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 2 ###

flights_df.dest.unique()
flights_df.head(10)

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_3.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 3 ###

flights_df["dest"][flights_df["dest"] == "SEA"].value_counts()

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_4.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 4 ###

flights_df["carrier"][flights_df["dest"] == "SEA"].value_counts()

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_5.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 5 ###

len(flights_df["tailnum"][flights_df["dest"] == "SEA"].unique())

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_6.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 6 ###

flights_df["arr_delay"][flights_df["dest"] == "SEA"].mean()

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_7.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 7 ###

f = flights_df[flights_df["dest"] == "SEA"].groupby("origin").size()
f_total = len(flights_df[flights_df.dest == "SEA"])
f.loc["EWR"] / f_total, f.loc["JFK"] / f_total

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_8.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 8 ###

df = flights_df.groupby(["month", "day"], as_index=False).agg({"dep_delay": np.mean})
df2 = flights_df.groupby(["month", "day"], as_index=False).agg({"arr_delay": np.mean})
df.loc[df["dep_delay"].idxmax()], df2.loc[df2["arr_delay"].idxmax()]

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_9.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 9 ###

df

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_10.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 10 ###

df = flights_df.groupby(["day", "month"], as_index=False).agg(
    {"arr_delay": np.mean, "dep_delay": np.mean}
)
df["total_delay"] = df["arr_delay"] + df["dep_delay"]
df.sort_values("total_delay", ascending=False).head(1)

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_11.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 11 ###

ds = flights_df.dropna(subset=["dep_delay"]).groupby(["month"])["dep_delay"].mean()
ds

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_12.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 12 ###

dt = flights_df.dropna(subset=["dep_delay"]).groupby(["hour"])["dep_delay"].mean()
dt

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_13.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 13 ###

df = flights_df
df["speed"] = df["distance"] / df["air_time"]
df[df["speed"] == df.speed.max()]

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_14.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 14 ###

count = len(flights_df)
df = flights_df.groupby(["carrier", "flight", "dest"]).size().reset_index(name="Size")
for i in df.index:
    if df.loc[i]["Size"] == 365:
        print(
            "Carrier: %s, Flight: %s, Destination: %s"
            % (df.loc[i]["carrier"], df.loc[i]["flight"], df.loc[i]["dest"])
        )

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_15.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 15 ###

gdf = flights_df
counts = gdf.groupby(["carrier", "flight", "dest"]).size().reset_index(name="size")
complete_year = counts[counts["size"] == 365]
complete_year[["carrier", "flight", "dest"]]

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_16.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 16 ###

df = flights_df[flights_df["month"] == 6]
df = df.groupby("carrier", as_index=False).agg(
    {"arr_delay": np.mean, "dep_delay": np.mean}
)
df["total_delay"] = df["arr_delay"] + df["dep_delay"]
for i in df.index:
    if df.loc[i]["total_delay"] == df["total_delay"].min():
        print(df.loc[i]["carrier"], df.loc[i]["total_delay"])
df

In [ ]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-flight/src/small_bench/checkpoints/pre_cell_17.pickle

In [ ]:
%%RecordEvent
%%cudf.pandas.profile
### cell 17 ###

weather_df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "nyc_weather.csv"
)
df = flights_df
df_c = pd.merge(df, weather_df, on=["month", "day", "hour", "origin"])
df_c.head(10)